In [1]:
import jax
import jax.numpy as jnp
from jax import lax


@jax.jit
def solve_eta(scattering_length, t_final, kernel_prefactor):
    scattering_length = jnp.asarray(scattering_length, dtype=jnp.complex64)

    num_points = scattering_length.shape[0]
    num_steps = num_points - 1
    dt = t_final / num_steps

    eta_0 = -4.0 * jnp.pi * scattering_length[0]
    eta = jnp.zeros_like(scattering_length, dtype=jnp.complex64).at[0].set(eta_0)

    l1_prefactor = 2.0 * kernel_prefactor / jnp.sqrt(dt)
    j = jnp.arange(num_steps)

    def time_step(history, k):
        diffs = history[1:] - history[:-1]

        # Only j = 0, ..., k - 2 is known. The j = k - 1 term contains eta[k]
        # and is moved into the denominator below.
        valid = j < (k - 1)
        m = k - j
        safe_m = jnp.maximum(m, 1)
        weights = jnp.where(
            valid,
            jnp.sqrt(safe_m) - jnp.sqrt(safe_m - 1),
            0.0,
        )

        known_history = jnp.sum(weights * diffs)
        numerator = -1.0 + l1_prefactor * (history[k - 1] - known_history)
        denominator = 1.0 / (4.0 * jnp.pi * scattering_length[k]) + l1_prefactor
        eta_k = numerator / denominator

        history = history.at[k].set(eta_k)
        return history, eta_k

    indices = jnp.arange(1, num_points)
    eta, _ = lax.scan(time_step, eta, indices)

    return eta


nondimensional_prefactor = -1.0 / (4.0 * jnp.pi ** (3 / 2) * jnp.sqrt(1j))
t_final = 1.0
num_steps = 10
time_grid = jnp.linspace(0.0, t_final, num_steps + 1)

scattering_length_1 = jnp.full(num_steps + 1, 0.20)
scattering_length_2 = jnp.linspace(0.10, 0.50, num_steps + 1)
scattering_length_3 = 0.20 + 0.05 * jnp.sin(2.0 * jnp.pi * time_grid)
scattering_length_4 = 0.20 + 0.03j * jnp.linspace(0.0, 1.0, num_steps + 1)
scattering_length_5 = jnp.linspace(0.12 + 0.02j, 0.42 + 0.08j, num_steps + 1)

test_cases = {
    "constant real": scattering_length_1,
    "ramped real": scattering_length_2,
    "oscillating real": scattering_length_3,
    "constant real plus imaginary ramp": scattering_length_4,
    "complex ramp": scattering_length_5,
}

print("scattering_length_1 eta:", solve_eta(test_cases["constant real"], t_final, nondimensional_prefactor))
print("scattering_length_2 eta:", solve_eta(test_cases["ramped real"], t_final, nondimensional_prefactor))
print("scattering_length_3 eta:", solve_eta(test_cases["oscillating real"], t_final, nondimensional_prefactor))
print("scattering_length_4 eta:", solve_eta(test_cases["constant real plus imaginary ramp"], t_final, nondimensional_prefactor))
print("scattering_length_5 eta:", solve_eta(test_cases["complex ramp"], t_final, nondimensional_prefactor))


scattering_length_1 eta: [-2.5132742+0.0000000e+00j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j]
scattering_length_2 eta: [-1.2566371+0.j         -1.855258 +0.3269458j  -2.4885173+0.57386273j
 -3.0798283+0.8436657j  -3.6475415+1.1604515j  -4.2010794+1.5181482j
 -4.7407727+1.9082669j  -5.264412 +2.3247974j  -5.7699437+2.763677j
 -6.25606  +3.2218826j  -6.722118 +3.6969175j ]
scattering_length_3 eta: [-2.5132742+0.j         -2.8168647+0.4171497j  -3.2752187+0.56529623j
 -3.5118535+0.2523072j  -3.1887152-0.30391625j -2.4050713-0.5455885j
 -1.8300257-0.3455062j  -1.6925277-0.15998557j -1.793363 -0.05399901j
 -2.0976427+0.11223325j -2.547984 +0.34507558j]
scattering_length_4 eta: [-2.5132742+0.j         -2.551909 -0.03791558j -2.5681593-0.10008954j
 -2.5656753-0.15293597j -